# Article 1 — Colab setup and Stage-0 verification

This notebook performs no data or model download beyond the project's Python dependencies. It is safe to run on a CPU runtime. Enter a repository URL only after the directory has been pushed to a remote Git host. Drive mounting is optional and should never expose credentials in the repository.

In [ ]:
# User-editable parameters
REPO_URL = 'PASTE_YOUR_GIT_REPOSITORY_URL_HERE'
BRANCH = 'main'
MOUNT_DRIVE = False
DRIVE_ROOT = '/content/drive/MyDrive/llm-rag-research/article1'
WORKDIR = '/content/rag-evidence-robustness'


In [ ]:
import os, platform, subprocess, sys
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    print(subprocess.check_output(['nvidia-smi'], text=True, stderr=subprocess.STDOUT)[:1000])
except (FileNotFoundError, subprocess.CalledProcessError):
    print('No NVIDIA GPU exposed by this runtime; CPU mode is expected to work.')

if REPO_URL.startswith('PASTE_'):
    raise ValueError('Set REPO_URL to the remote repository URL before running this cell.')
if not Path(WORKDIR).exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, WORKDIR], check=True)
%cd /content/rag-evidence-robustness


In [ ]:
# Editable install plus test tooling. No GPU is required.
!{sys.executable} -m pip install -e '.[dev]'
!{sys.executable} -m pytest -q
!{sys.executable} -m evidence_robustness.cli smoke --config configs/smoke.yaml


In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    root = Path(DRIVE_ROOT)
    for relative in ['datasets/raw', 'datasets/processed', 'datasets/perturbations', 'features', 'models', 'runs', 'reports']:
        (root / relative).mkdir(parents=True, exist_ok=True)
    print('Created/verified Drive layout:', root)
else:
    print('Drive was not mounted. This does not affect Stage-0 tests.')


## What to report back

Send the complete output of `pytest -q` and the JSON emitted by `smoke`. If Drive was mounted, also report whether the listed folders were created. Do not report a GPU as available unless `nvidia-smi` actually printed one.